# Credit Risk Analysis: End-to-End Data Analytics Pipeline

A complete walkthrough: data cleaning, exploratory data analysis, statistical
visualization, machine learning default-risk models, portfolio forecasting,
and a SQL analytics layer, built on the `credit_risk_dataset` (32,581 loan
records, 12 raw fields).

**Pipeline:** Data Cleaning → EDA & Visualization (24 charts) → Predictive
Modeling (3 classifiers) → Forecasting (Holt's linear trend) → SQL Dashboard
Layer (validated against SQLite)

This notebook combines all 7 pipeline scripts into one narrative file. All
outputs (charts, printed results) below were captured from an actual run of
the pipeline, this notebook renders fully on GitHub without needing to be
re-executed, and can also be run top-to-bottom locally.

---


## 0. Setup

In [74]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                              classification_report, precision_recall_curve, average_precision_score)
import joblib, sqlite3, os

BASE = r"C:\Users\Administrator\Desktop\GitHub Repository\Credit Risk Analysis"

for sub in ["data", "visuals", "models", "sql", "sql/query_results"]:
    os.makedirs(os.path.join(BASE, sub), exist_ok=True)

os.chdir(BASE)
print("Working directory set to:", os.getcwd())

# Place credit_risk_dataset.csv directly inside BASE (or inside BASE/data —
# just make sure RAW_PATH below matches wherever it actually is)
RAW_PATH = "credit_risk_dataset.csv"

sns.set_theme(style="whitegrid", font_scale=1.05)
PALETTE = ["#2E5E8C", "#E8743B", "#3AA655", "#C94C4C", "#8E6FBE", "#D4AC0D"]
sns.set_palette(PALETTE)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.titleweight"] = "bold"


Working directory set to: C:\Users\Administrator\Desktop\GitHub Repository\Credit Risk Analysis


## 1. Data Cleaning

Raw dataset: **32,581 rows, 12 columns**. Issues found during profiling:
- 165 exact duplicate rows
- Implausible outliers: age up to 144, employment length up to 123 years, income up to $6,000,000
- Missing values: 895 in `person_emp_length`, 3,116 in `loan_int_rate`

The cleaning strategy below removes logically-impossible outliers, imputes
missing values using **contextual medians** (age-bracket for employment
length, loan-grade for interest rate, since rate is fundamentally grade-driven
rather than random), and engineers 7 new analytical features.

In [39]:
RAW_PATH = "credit_risk_dataset.csv"
OUT_PATH = "data/credit_risk_clean.csv"

def load_data(path):
    df = pd.read_csv(path)
    print(f"Raw shape: {df.shape}")
    return df

def clean(df):
    log = []

    # 1. Drop exact duplicate rows
    before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    log.append(f"Removed {before - len(df)} duplicate rows")

    # 2. Remove logically impossible outliers
    # Age: keep realistic working-age borrowers (18-80)
    before = len(df)
    df = df[(df.person_age >= 18) & (df.person_age <= 80)]
    log.append(f"Removed {before - len(df)} rows with implausible age (<18 or >80)")

    # Employment length: cannot exceed working years given age; cap at 60
    before = len(df)
    df = df[(df.person_emp_length.isna()) | (df.person_emp_length <= 60)]
    log.append(f"Removed {before - len(df)} rows with implausible emp_length (>60 yrs)")

    # Employment length can't exceed person_age - 14 (legal working age assumption)
    before = len(df)
    df = df[(df.person_emp_length.isna()) | (df.person_emp_length <= (df.person_age - 14))]
    log.append(f"Removed {before - len(df)} rows where emp_length exceeds plausible working years")

    # Income: cap extreme outliers (>$1M) - likely data entry errors for this population
    before = len(df)
    df = df[df.person_income <= 1_000_000]
    log.append(f"Removed {before - len(df)} rows with income > $1,000,000 (extreme outliers)")

    # 3. Handle missing values
    # person_emp_length: impute with median grouped by age bracket
    missing_emp = df.person_emp_length.isna().sum()
    df["person_emp_length"] = df.groupby(pd.cut(df.person_age, bins=[17,25,35,50,80]),observed=True)["person_emp_length"]\
        .transform(lambda x: x.fillna(x.median()))
    df["person_emp_length"] = df["person_emp_length"].fillna(df.person_emp_length.median())
    log.append(f"Imputed {missing_emp} missing person_emp_length values (median by age bracket)")

    # loan_int_rate: impute with median grouped by loan_grade (rate is grade-driven)
    missing_rate = df.loan_int_rate.isna().sum()
    df["loan_int_rate"] = df.groupby("loan_grade")["loan_int_rate"].transform(lambda x: x.fillna(x.median()))
    log.append(f"Imputed {missing_rate} missing loan_int_rate values (median by loan_grade)")

    # 4. Dtype fixes
    df["person_emp_length"] = df["person_emp_length"].round(1)
    df["loan_int_rate"] = df["loan_int_rate"].round(2)

    # 5. Feature engineering
    df["age_group"] = pd.cut(df.person_age, bins=[17,25,35,45,55,80],
                               labels=["18-25","26-35","36-45","46-55","56+"])
    df["income_bracket"] = pd.cut(df.person_income,
                                    bins=[0,25000,50000,75000,100000,150000,np.inf],
                                    labels=["<25K","25-50K","50-75K","75-100K","100-150K","150K+"])
    df["loan_to_income_pct"] = (df.loan_amnt / df.person_income * 100).round(2)
    df["dti_risk_band"] = pd.cut(df.loan_percent_income, bins=[-0.01,0.1,0.2,0.3,0.4,1.0],
                                   labels=["Very Low","Low","Moderate","High","Very High"])
    df["loan_status_label"] = df.loan_status.map({0:"Non-Default", 1:"Default"})
    df["has_prior_default"] = df.cb_person_default_on_file.map({"Y":1,"N":0})

    # credit history to age ratio (maturity of credit profile)
    df["credit_hist_ratio"] = (df.cb_person_cred_hist_length / (df.person_age - 17)).round(2)

    df = df.reset_index(drop=True)
    log.append(f"Final clean shape: {df.shape}")

    print("\n".join(log))
    return df, log

df = load_data(RAW_PATH)
clean_df, log = clean(df)
clean_df.to_csv(OUT_PATH, index=False)
with open("data/cleaning_log.txt", "w") as f:
    f.write("CREDIT RISK DATASET - CLEANING LOG\n" + "="*40 + "\n")
    f.write("\n".join(log))
print(f"\nSaved cleaned dataset to {OUT_PATH}")
print(f"\nMissing values remaining:\n{clean_df.isnull().sum()[clean_df.isnull().sum()>0]}")


Raw shape: (32399, 19)
Removed 0 duplicate rows
Removed 0 rows with implausible age (<18 or >80)
Removed 0 rows with implausible emp_length (>60 yrs)
Removed 0 rows where emp_length exceeds plausible working years
Removed 0 rows with income > $1,000,000 (extreme outliers)
Imputed 0 missing person_emp_length values (median by age bracket)
Imputed 0 missing loan_int_rate values (median by loan_grade)
Final clean shape: (32399, 19)

Saved cleaned dataset to data/credit_risk_clean.csv

Missing values remaining:
Series([], dtype: int64)


## 2. Exploratory Data Analysis & Visualization

24 charts total across this notebook, saved individually to `visuals/` by
the pipeline scripts. Each block below reproduces one chart or chart group.

In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

DATA_PATH = "data/credit_risk_clean.csv"
VIZ_DIR = "visuals"

# ---- Global style ----
sns.set_theme(style="whitegrid", font_scale=1.05)
PALETTE = ["#2E5E8C", "#E8743B", "#3AA655", "#C94C4C", "#8E6FBE", "#D4AC0D"]
sns.set_palette(PALETTE)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelsize"] = 11

df = pd.read_csv(DATA_PATH)
order_grade = sorted(df.loan_grade.unique())
order_age = ["18-25","26-35","36-45","46-55","56+"]
order_income = ["<25K","25-50K","50-75K","75-100K","100-150K","150K+"]
order_dti = ["Very Low","Low","Moderate","High","Very High"]

def save(fig, name):
    fig.tight_layout()
    fig.savefig(f"{VIZ_DIR}/{name}.png", bbox_inches="tight")
    plt.close(fig)
    print(f"saved {name}.png")

### 2.1 Target Variable Distribution
Checking class balance before modeling, this is a moderately imbalanced problem (~22% defaults).

In [41]:
fig, ax = plt.subplots(figsize=(6,5))
counts = df.loan_status_label.value_counts()
colors = ["#2E5E8C", "#C94C4C"]
bars = ax.bar(counts.index, counts.values, color=colors, width=0.55)
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+200, f"{b.get_height():,}\n({b.get_height()/len(df)*100:.1f}%)",
            ha="center", fontsize=11, fontweight="bold")
ax.set_title("Loan Status Distribution: Class Imbalance Check")
ax.set_ylabel("Number of Loans")
ax.set_xlabel("")
save(fig, "01_target_distribution")

saved 01_target_distribution.png


### 2.2 Numeric Feature Distributions

In [42]:
num_cols = ["person_age","person_income","person_emp_length","loan_amnt",
            "loan_int_rate","loan_percent_income","cb_person_cred_hist_length"]
fig, axes = plt.subplots(3, 3, figsize=(16,12))
axes = axes.flatten()
titles = ["Age (years)","Annual Income ($)","Employment Length (yrs)","Loan Amount ($)",
          "Interest Rate (%)","Loan-to-Income Ratio","Credit History Length (yrs)"]
for i, (col, t) in enumerate(zip(num_cols, titles)):
    sns.histplot(df[col], bins=40, ax=axes[i], color=PALETTE[0], kde=True)
    axes[i].set_title(t)
    axes[i].set_xlabel("")
for j in range(len(num_cols), len(axes)):
    fig.delaxes(axes[j])
fig.suptitle("Distribution of Key Numeric Features", fontsize=16, fontweight="bold", y=1.02)
save(fig, "02_numeric_distributions")

saved 02_numeric_distributions.png


### 2.3 Correlation Matrix

In [43]:
corr_cols = ["person_age","person_income","person_emp_length","loan_amnt",
             "loan_int_rate","loan_percent_income","cb_person_cred_hist_length",
             "loan_status","has_prior_default"]
corr = df[corr_cols].corr()
fig, ax = plt.subplots(figsize=(9,7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink":0.8}, ax=ax)
ax.set_title("Correlation Matrix: Numeric Features & Default Risk")
save(fig, "03_correlation_heatmap")

saved 03_correlation_heatmap.png


### 2.4 Default Rate by Loan Grade
Default rate climbs from 10% (Grade A) to 98% (Grade G), a clean monotonic risk gradient.

In [44]:
grade_default = df.groupby("loan_grade")["loan_status"].agg(["mean","count"]).reindex(order_grade)
fig, ax1 = plt.subplots(figsize=(9,6))
bars = ax1.bar(grade_default.index, grade_default["mean"]*100, color=PALETTE[1], alpha=0.85)
ax1.set_ylabel("Default Rate (%)", color=PALETTE[1], fontweight="bold")
ax1.set_xlabel("Loan Grade")
for b, v in zip(bars, grade_default["mean"]*100):
    ax1.text(b.get_x()+b.get_width()/2, v+1, f"{v:.1f}%", ha="center", fontweight="bold")
ax2 = ax1.twinx()
ax2.plot(grade_default.index, grade_default["count"], color=PALETTE[0], marker="o", linewidth=2.5, markersize=8)
ax2.set_ylabel("Number of Loans", color=PALETTE[0], fontweight="bold")
ax2.grid(False)
ax1.set_title("Default Rate & Volume by Loan Grade")
save(fig, "04_default_rate_by_grade")

saved 04_default_rate_by_grade.png


### 2.5 Default Rate by Loan Purpose

In [45]:
intent_default = df.groupby("loan_intent")["loan_status"].mean().sort_values(ascending=False)*100
fig, ax = plt.subplots(figsize=(9,6))
bars = ax.barh(intent_default.index[::-1], intent_default.values[::-1], color=PALETTE[2])
for b, v in zip(bars, intent_default.values[::-1]):
    ax.text(v+0.3, b.get_y()+b.get_height()/2, f"{v:.1f}%", va="center", fontweight="bold")
ax.set_xlabel("Default Rate (%)")
ax.set_title("Default Rate by Loan Purpose")
save(fig, "05_default_rate_by_intent")

saved 05_default_rate_by_intent.png


### 2.6 Default Rate by Home Ownership

In [46]:
home_default = df.groupby("person_home_ownership")["loan_status"].mean().sort_values(ascending=False)*100
fig, ax = plt.subplots(figsize=(7,5.5))
bars = ax.bar(home_default.index, home_default.values, color=PALETTE[3])
for b, v in zip(bars, home_default.values):
    ax.text(b.get_x()+b.get_width()/2, v+0.5, f"{v:.1f}%", ha="center", fontweight="bold")
ax.set_ylabel("Default Rate (%)")
ax.set_title("Default Rate by Home Ownership Status")
save(fig, "06_default_rate_by_home_ownership")

print("Batch 1 complete.")

saved 06_default_rate_by_home_ownership.png
Batch 1 complete.


In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = "data/credit_risk_clean.csv"
VIZ_DIR = "visuals"

sns.set_theme(style="whitegrid", font_scale=1.05)
PALETTE = ["#2E5E8C", "#E8743B", "#3AA655", "#C94C4C", "#8E6FBE", "#D4AC0D"]
sns.set_palette(PALETTE)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.titleweight"] = "bold"

df = pd.read_csv(DATA_PATH)
order_grade = sorted(df.loan_grade.unique())
order_age = ["18-25","26-35","36-45","46-55","56+"]
order_income = ["<25K","25-50K","50-75K","75-100K","100-150K","150K+"]
order_dti = ["Very Low","Low","Moderate","High","Very High"]
status_colors = {"Non-Default": "#2E5E8C", "Default": "#C94C4C"}

def save(fig, name):
    fig.tight_layout()
    fig.savefig(f"{VIZ_DIR}/{name}.png", bbox_inches="tight")
    plt.close(fig)
    print(f"saved {name}.png")

### 2.7 Feature Spread: Default vs Non-Default

In [48]:
fig, axes = plt.subplots(2, 3, figsize=(16,9))
box_cols = ["person_age","person_income","loan_amnt","loan_int_rate","loan_percent_income","cb_person_cred_hist_length"]
box_titles = ["Age","Income","Loan Amount","Interest Rate","Loan/Income Ratio","Credit History Length"]
for ax, col, t in zip(axes.flatten(), box_cols, box_titles):
    sns.boxplot(data=df, x="loan_status_label", y=col, ax=ax, palette=status_colors, hue="loan_status_label", legend=False)
    ax.set_title(t)
    ax.set_xlabel("")
fig.suptitle("Feature Spread & Outliers: Default vs Non-Default", fontsize=16, fontweight="bold", y=1.02)
save(fig, "07_boxplots_by_default_status")

saved 07_boxplots_by_default_status.png


### 2.8 Income vs Loan Amount

In [49]:
fig, ax = plt.subplots(figsize=(9,7))
sample = df.sample(min(6000, len(df)), random_state=42)
for status, c in status_colors.items():
    sub = sample[sample.loan_status_label == status]
    ax.scatter(sub.person_income, sub.loan_amnt, alpha=0.35, s=18, color=c, label=status)
ax.set_xlim(0, 200000)
ax.set_xlabel("Annual Income ($)")
ax.set_ylabel("Loan Amount ($)")
ax.set_title("Income vs Loan Amount, Colored by Default Status")
ax.legend(title="Loan Status")
save(fig, "08_income_vs_loanamount_scatter")

saved 08_income_vs_loanamount_scatter.png


### 2.9 Risk Matrix: Grade × Home Ownership

In [50]:
pivot = df.pivot_table(values="loan_status", index="loan_grade", columns="person_home_ownership",
                        aggfunc="mean", observed=True).reindex(order_grade) * 100
fig, ax = plt.subplots(figsize=(8,6.5))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="OrRd", cbar_kws={"label":"Default Rate (%)"}, ax=ax, linewidths=0.5)
ax.set_title("Default Rate (%): Loan Grade x Home Ownership")
ax.set_xlabel("Home Ownership")
ax.set_ylabel("Loan Grade")
save(fig, "09_heatmap_grade_vs_homeownership")

saved 09_heatmap_grade_vs_homeownership.png


### 2.10 Default Rate by Age Group & Income Bracket

In [51]:
fig, axes = plt.subplots(1, 2, figsize=(15,6))
age_default = df.groupby("age_group", observed=True)["loan_status"].mean().reindex(order_age)*100
axes[0].bar(age_default.index, age_default.values, color=PALETTE[4])
for i,v in enumerate(age_default.values):
    axes[0].text(i, v+0.5, f"{v:.1f}%", ha="center", fontweight="bold")
axes[0].set_title("Default Rate by Age Group")
axes[0].set_ylabel("Default Rate (%)")

inc_default = df.groupby("income_bracket", observed=True)["loan_status"].mean().reindex(order_income)*100
axes[1].bar(inc_default.index, inc_default.values, color=PALETTE[5])
for i,v in enumerate(inc_default.values):
    axes[1].text(i, v+0.5, f"{v:.1f}%", ha="center", fontweight="bold")
axes[1].set_title("Default Rate by Income Bracket")
axes[1].set_ylabel("Default Rate (%)")
axes[1].tick_params(axis='x', rotation=20)
save(fig, "10_default_by_age_income_bracket")

saved 10_default_by_age_income_bracket.png


### 2.11 Default Rate by Debt-to-Income Band

In [52]:
dti_default = df.groupby("dti_risk_band", observed=True)["loan_status"].agg(["mean","count"]).reindex(order_dti)
fig, ax1 = plt.subplots(figsize=(9,6))
bars = ax1.bar(dti_default.index, dti_default["mean"]*100, color=PALETTE[3], alpha=0.85)
for b,v in zip(bars, dti_default["mean"]*100):
    ax1.text(b.get_x()+b.get_width()/2, v+1, f"{v:.1f}%", ha="center", fontweight="bold")
ax1.set_ylabel("Default Rate (%)")
ax1.set_xlabel("Debt-to-Income Risk Band")
ax1.set_title("Default Rate by Loan-to-Income (DTI) Risk Band")
save(fig, "11_default_by_dti_band")

saved 11_default_by_dti_band.png


### 2.12 Prior Default History Impact
Borrowers with a prior bureau default show a markedly higher current default rate.

In [53]:
fig, ax = plt.subplots(figsize=(7,5.5))
prior_default = df.groupby("cb_person_default_on_file")["loan_status"].mean()*100
labels = ["No Prior Default", "Prior Default on File"]
bars = ax.bar(labels, [prior_default["N"], prior_default["Y"]], color=[PALETTE[0], PALETTE[3]])
for b,v in zip(bars, [prior_default["N"], prior_default["Y"]]):
    ax.text(b.get_x()+b.get_width()/2, v+0.5, f"{v:.1f}%", ha="center", fontweight="bold")
ax.set_ylabel("Current Default Rate (%)")
ax.set_title("Impact of Prior Default History on Current Default Rate")
save(fig, "12_prior_default_impact")

saved 12_prior_default_impact.png


### 2.13 Interest Rate Distribution by Grade

In [54]:
fig, ax = plt.subplots(figsize=(10,6))
sns.violinplot(data=df, x="loan_grade", y="loan_int_rate", order=order_grade, ax=ax,
                palette=PALETTE, hue="loan_grade", legend=False)
ax.set_title("Interest Rate Distribution by Loan Grade")
ax.set_xlabel("Loan Grade")
ax.set_ylabel("Interest Rate (%)")
save(fig, "13_interest_rate_by_grade_violin")

C:\Users\Administrator\AppData\Local\Temp\ipykernel_5040\4035878868.py:2: UserWarning: 
The palette list has fewer values (6) than needed (7) and will cycle, which may produce an uninterpretable plot.
  sns.violinplot(data=df, x="loan_grade", y="loan_int_rate", order=order_grade, ax=ax,


saved 13_interest_rate_by_grade_violin.png


### 2.14 Portfolio Mix & Grade Volume

In [55]:
fig, axes = plt.subplots(1,2, figsize=(14,6))
intent_counts = df.loan_intent.value_counts()
axes[0].pie(intent_counts.values, labels=intent_counts.index, autopct="%1.1f%%",
            colors=PALETTE, wedgeprops=dict(width=0.4), pctdistance=0.8, startangle=90)
axes[0].set_title("Loan Portfolio Mix by Intent")

grade_counts = df.loan_grade.value_counts().reindex(order_grade)
axes[1].bar(grade_counts.index, grade_counts.values, color=PALETTE[0])
axes[1].set_title("Loan Volume by Grade")
axes[1].set_ylabel("Number of Loans")
save(fig, "14_portfolio_mix_and_grade_volume")

saved 14_portfolio_mix_and_grade_volume.png


### 2.15 Pairwise Feature Relationships

In [56]:
pair_cols = ["person_age","person_income","loan_amnt","loan_int_rate","loan_percent_income","loan_status_label"]
g = sns.pairplot(df[pair_cols].sample(min(3000,len(df)), random_state=1),
                  hue="loan_status_label", palette=status_colors, diag_kind="kde",
                  plot_kws={"alpha":0.4, "s":15})
g.fig.suptitle("Pairwise Feature Relationships by Default Status", y=1.02, fontsize=16, fontweight="bold")
g.fig.savefig(f"{VIZ_DIR}/15_pairplot.png", bbox_inches="tight", dpi=150)
plt.close(g.fig)
print("saved 15_pairplot.png")

print("Batch 2 complete.")

saved 15_pairplot.png
Batch 2 complete.


## 3. Predictive Modeling: Default Risk Scoring

Three classifiers are trained to predict `loan_status` (default probability):
Logistic Regression (baseline, interpretable), Random Forest, and Gradient
Boosting (best performer). Categorical features are label-encoded; numeric
features are standardized for the logistic model.

In [57]:
DATA_PATH = "data/credit_risk_clean.csv"
VIZ_DIR = "visuals"
MODEL_DIR = "models"

df = pd.read_csv(DATA_PATH)

### Feature prep

In [58]:
features = ["person_age","person_income","person_emp_length","loan_amnt",
            "loan_int_rate","loan_percent_income","cb_person_cred_hist_length",
            "has_prior_default","person_home_ownership","loan_intent","loan_grade"]
target = "loan_status"

X = df[features].copy()
y = df[target].copy()

cat_cols = ["person_home_ownership","loan_intent","loan_grade"]
encoders = {}
for c in cat_cols:
    le = LabelEncoder()
    X[c] = le.fit_transform(X[c])
    encoders[c] = le

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Models

In [59]:
log_reg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
log_reg.fit(X_train_scaled, y_train)

rf = RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=5,
                              class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

gb = GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)

models = {
    "Logistic Regression": (log_reg, X_test_scaled),
    "Random Forest": (rf, X_test),
    "Gradient Boosting": (gb, X_test),
}

results = {}
for name, (model, X_te) in models.items():
    y_prob = model.predict_proba(X_te)[:,1]
    y_pred = model.predict(X_te)
    auc = roc_auc_score(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    results[name] = {"model": model, "y_prob": y_prob, "y_pred": y_pred, "auc": auc, "ap": ap}
    print(f"\n{name}: ROC-AUC = {auc:.4f} | PR-AUC = {ap:.4f}")
    print(classification_report(y_test, y_pred, target_names=["Non-Default","Default"]))

best_name = max(results, key=lambda k: results[k]["auc"])
print(f"\nBest model: {best_name} (ROC-AUC={results[best_name]['auc']:.4f})")


Logistic Regression: ROC-AUC = 0.8572 | PR-AUC = 0.6713
              precision    recall  f1-score   support

 Non-Default       0.93      0.80      0.86      5062
     Default       0.52      0.78      0.62      1418

    accuracy                           0.79      6480
   macro avg       0.72      0.79      0.74      6480
weighted avg       0.84      0.79      0.81      6480


Random Forest: ROC-AUC = 0.9282 | PR-AUC = 0.8780
              precision    recall  f1-score   support

 Non-Default       0.93      0.96      0.94      5062
     Default       0.83      0.75      0.79      1418

    accuracy                           0.91      6480
   macro avg       0.88      0.86      0.87      6480
weighted avg       0.91      0.91      0.91      6480


Gradient Boosting: ROC-AUC = 0.9434 | PR-AUC = 0.8970
              precision    recall  f1-score   support

 Non-Default       0.92      0.99      0.96      5062
     Default       0.96      0.71      0.82      1418

    accuracy       

### 3.1 ROC Curve Comparison

In [60]:
fig, ax = plt.subplots(figsize=(8,7))
colors = [PALETTE[0], PALETTE[1], PALETTE[2]]
for (name, r), c in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, r["y_prob"])
    ax.plot(fpr, tpr, label=f"{name} (AUC={r['auc']:.3f})", color=c, linewidth=2.5)
ax.plot([0,1],[0,1], "k--", alpha=0.4, label="Random Baseline")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve Comparison: Default Prediction Models")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(f"{VIZ_DIR}/16_roc_curve_comparison.png", bbox_inches="tight")
plt.close(fig)
print("saved 16_roc_curve_comparison.png")

saved 16_roc_curve_comparison.png


### 3.2 Confusion Matrix (Best Model)

In [61]:
best = results[best_name]
cm = confusion_matrix(y_test, best["y_pred"])
fig, ax = plt.subplots(figsize=(6.5,5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Non-Default","Default"], yticklabels=["Non-Default","Default"], ax=ax,
            annot_kws={"size":14, "weight":"bold"})
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix - {best_name}")
fig.tight_layout()
fig.savefig(f"{VIZ_DIR}/17_confusion_matrix.png", bbox_inches="tight")
plt.close(fig)
print("saved 17_confusion_matrix.png")

saved 17_confusion_matrix.png


### 3.3 Feature Importance\n`loan_percent_income` (loan-to-income ratio) is the single strongest predictor of default, followed by `loan_grade` and `person_income`.

In [62]:
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9,7))
bars = ax.barh(importances.index, importances.values, color=PALETTE[0])
for b, v in zip(bars, importances.values):
    ax.text(v+0.002, b.get_y()+b.get_height()/2, f"{v:.3f}", va="center", fontsize=9)
ax.set_title("Feature Importance for Default Prediction (Random Forest)")
ax.set_xlabel("Importance")
fig.tight_layout()
fig.savefig(f"{VIZ_DIR}/18_feature_importance.png", bbox_inches="tight")
plt.close(fig)
print("saved 18_feature_importance.png")

saved 18_feature_importance.png


### 3.4 Predicted Default Probability Distribution

In [63]:
fig, ax = plt.subplots(figsize=(9,6))
prob_df = pd.DataFrame({"prob": best["y_prob"], "actual": y_test.values})
sns.histplot(data=prob_df, x="prob", hue="actual", bins=40, palette=[PALETTE[0],PALETTE[3]],
             element="step", stat="density", common_norm=False, ax=ax)
ax.set_xlabel("Predicted Probability of Default")
ax.set_title(f"Predicted Default Probability Distribution - {best_name}")
ax.legend(labels=["Default","Non-Default"], title="Actual")
fig.tight_layout()
fig.savefig(f"{VIZ_DIR}/19_predicted_probability_distribution.png", bbox_inches="tight")
plt.close(fig)
print("saved 19_predicted_probability_distribution.png")

saved 19_predicted_probability_distribution.png


### 3.5 Precision-Recall Curve Comparison

In [64]:
fig, ax = plt.subplots(figsize=(8,7))
for (name, r), c in zip(results.items(), colors):
    prec, rec, _ = precision_recall_curve(y_test, r["y_prob"])
    ax.plot(rec, prec, label=f"{name} (AP={r['ap']:.3f})", color=c, linewidth=2.5)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve Comparison")
ax.legend(loc="lower left")
fig.tight_layout()
fig.savefig(f"{VIZ_DIR}/20_precision_recall_curve.png", bbox_inches="tight")
plt.close(fig)
print("saved 20_precision_recall_curve.png")

saved 20_precision_recall_curve.png


### Save best model + scoring artifacts

In [65]:
joblib.dump(rf, f"{MODEL_DIR}/random_forest_model.pkl")
joblib.dump(gb, f"{MODEL_DIR}/gradient_boosting_model.pkl")
joblib.dump(scaler, f"{MODEL_DIR}/scaler.pkl")
joblib.dump(encoders, f"{MODEL_DIR}/label_encoders.pkl")

# Score full portfolio for BI export
X_full = df[features].copy()
for c in cat_cols:
    X_full[c] = encoders[c].transform(X_full[c])
df["predicted_default_probability"] = rf.predict_proba(X_full)[:,1].round(4)
df["predicted_risk_tier"] = pd.cut(df.predicted_default_probability, bins=[-0.01,0.2,0.4,0.6,0.8,1.0],
                                     labels=["Very Low","Low","Moderate","High","Very High"])
df.to_csv(DATA_PATH.replace("credit_risk_clean.csv","credit_risk_scored.csv"), index=False)

with open("models/model_summary.txt","w") as f:
    f.write("CREDIT RISK MODEL PERFORMANCE SUMMARY\n" + "="*45 + "\n\n")
    for name, r in results.items():
        f.write(f"{name}:\n  ROC-AUC: {r['auc']:.4f}\n  PR-AUC (Avg Precision): {r['ap']:.4f}\n\n")
    f.write(f"Best Model: {best_name}\n\n")
    f.write("Top 5 Features (Random Forest):\n")
    for feat, imp in importances.sort_values(ascending=False).head(5).items():
        f.write(f"  {feat}: {imp:.4f}\n")

print(f"\nScored dataset saved. Model summary saved.")
print("Batch 3 (modeling) complete.")


Scored dataset saved. Model summary saved.
Batch 3 (modeling) complete.


## 4. Forecasting

> **Methodology note:** this dataset is cross-sectional with no application-date
> field. To demonstrate time-series forecasting, a synthetic monthly application
> date is simulated (seeded/reproducible, mild seasonal weighting) across a
> 36-month window. The *forecasting method and trend shape* are the deliverable
> here, treat the specific historical dates as illustrative, not real
> transaction dates.

A hand-rolled **Holt's Linear Trend** (double exponential smoothing) model
forecasts the next 6 months of loan volume and default rate, with 95%
confidence bands derived from residual standard deviation.

In [66]:
import matplotlib.dates as mdates

DATA_PATH = "data/credit_risk_scored.csv"
VIZ_DIR = "visuals"

np.random.seed(42)
df = pd.read_csv(DATA_PATH)

# ---------------------------------------------------------------
# Simulate application dates (36-month window ending "current" month)
# Weighted slightly to mimic seasonal loan demand (higher in Q1/Q4)
# ---------------------------------------------------------------
n = len(df)
start = pd.Timestamp("2023-01-01")
months = pd.date_range(start, periods=36, freq="MS")
seasonal_weights = np.array([1.15,1.05,1.0,0.95,0.9,0.85,0.85,0.9,0.95,1.05,1.15,1.25]*3)
seasonal_weights = seasonal_weights / seasonal_weights.sum()
month_choices = np.random.choice(months, size=n, p=seasonal_weights)
day_choices = np.random.randint(1,28,size=n)
df["sim_application_date"] = pd.to_datetime(month_choices) + pd.to_timedelta(day_choices, unit="D")
df["sim_year_month"] = df["sim_application_date"].dt.to_period("M").dt.to_timestamp()

monthly = df.groupby("sim_year_month").agg(
    loan_volume=("loan_status","size"),
    default_rate=("loan_status","mean"),
    avg_loan_amt=("loan_amnt","mean")
).reset_index()
monthly["default_rate_pct"] = monthly["default_rate"]*100

def save(fig, name):
    fig.tight_layout()
    fig.savefig(f"{VIZ_DIR}/{name}.png", bbox_inches="tight")
    plt.close(fig)
    print(f"saved {name}.png")

### Holt's Linear Trend (hand-rolled double exponential smoothing)

In [67]:
def holt_linear(series, alpha=0.4, beta=0.2, periods_ahead=6):
    level = series[0]
    trend = series[1] - series[0]
    levels, trends = [level], [trend]
    for t in range(1, len(series)):
        last_level = level
        level = alpha*series[t] + (1-alpha)*(level+trend)
        trend = beta*(level-last_level) + (1-beta)*trend
        levels.append(level); trends.append(trend)
    forecast = [level + (i+1)*trend for i in range(periods_ahead)]
    return np.array(levels), np.array(forecast)

vol_series = monthly["loan_volume"].values.astype(float)
levels_v, forecast_v = holt_linear(vol_series, alpha=0.5, beta=0.25, periods_ahead=6)

dr_series = monthly["default_rate_pct"].values.astype(float)
levels_d, forecast_d = holt_linear(dr_series, alpha=0.4, beta=0.15, periods_ahead=6)

future_months = pd.date_range(monthly["sim_year_month"].max() + pd.offsets.MonthBegin(1), periods=6, freq="MS")

# simple bootstrap-based confidence band from residual std
resid_v = vol_series - levels_v
std_v = resid_v.std()
resid_d = dr_series - levels_d
std_d = resid_d.std()

### 4.1 Loan Application Volume: 6-Month Forecast

In [68]:
fig, ax = plt.subplots(figsize=(13,6.5))
ax.plot(monthly["sim_year_month"], monthly["loan_volume"], marker="o", color=PALETTE[0],
        linewidth=2, label="Historical Volume (simulated monthly)")
ax.plot(future_months, forecast_v, marker="o", linestyle="--", color=PALETTE[1],
        linewidth=2.5, label="Forecast (Holt's Linear Trend)")
upper = forecast_v + 1.96*std_v*np.sqrt(np.arange(1,7))
lower = forecast_v - 1.96*std_v*np.sqrt(np.arange(1,7))
ax.fill_between(future_months, lower, upper, color=PALETTE[1], alpha=0.15, label="95% Confidence Band")
ax.axvline(monthly["sim_year_month"].max(), color="gray", linestyle=":", alpha=0.7)
ax.set_title("Loan Application Volume: 6-Month Forecast\n(simulated monthly cohort — see methodology note)")
ax.set_ylabel("Number of Loan Applications")
ax.set_xlabel("Month")
ax.legend(loc="upper left")
fig.autofmt_xdate()
save(fig, "21_forecast_loan_volume")

saved 21_forecast_loan_volume.png


### 4.2 Portfolio Default Rate: 6-Month Forecast

In [69]:
fig, ax = plt.subplots(figsize=(13,6.5))
ax.plot(monthly["sim_year_month"], monthly["default_rate_pct"], marker="o", color=PALETTE[3],
        linewidth=2, label="Historical Default Rate (simulated monthly)")
ax.plot(future_months, forecast_d, marker="o", linestyle="--", color=PALETTE[4],
        linewidth=2.5, label="Forecast (Holt's Linear Trend)")
upper_d = forecast_d + 1.96*std_d*np.sqrt(np.arange(1,7))
lower_d = forecast_d - 1.96*std_d*np.sqrt(np.arange(1,7))
ax.fill_between(future_months, lower_d, upper_d, color=PALETTE[4], alpha=0.15, label="95% Confidence Band")
ax.axvline(monthly["sim_year_month"].max(), color="gray", linestyle=":", alpha=0.7)
ax.set_title("Portfolio Default Rate: 6-Month Forecast\n(simulated monthly cohort — see methodology note)")
ax.set_ylabel("Default Rate (%)")
ax.set_xlabel("Month")
ax.legend(loc="upper left")
fig.autofmt_xdate()
save(fig, "22_forecast_default_rate")

saved 22_forecast_default_rate.png


### 4.3 Current Portfolio Composition by Model-Predicted Risk Tier

In [70]:
tier_order = ["Very Low","Low","Moderate","High","Very High"]
tier_counts = df["predicted_risk_tier"].value_counts().reindex(tier_order)
fig, ax = plt.subplots(figsize=(9,6))
colors_tier = ["#2E5E8C","#3AA655","#D4AC0D","#E8743B","#C94C4C"]
bars = ax.bar(tier_counts.index, tier_counts.values, color=colors_tier)
for b,v in zip(bars, tier_counts.values):
    ax.text(b.get_x()+b.get_width()/2, v+150, f"{v:,}\n({v/len(df)*100:.1f}%)", ha="center", fontweight="bold")
ax.set_title("Current Portfolio Composition by Model-Predicted Risk Tier")
ax.set_ylabel("Number of Loans")
save(fig, "23_risk_tier_portfolio_composition")

# Save full dataset WITH simulated date (for SQL / Power BI use)
df.to_csv("data/credit_risk_full.csv", index=False)

# Save monthly + forecast tables for SQL / Power BI use
monthly.to_csv("data/monthly_portfolio_trend.csv", index=False)
forecast_table = pd.DataFrame({
    "forecast_month": future_months,
    "forecast_loan_volume": forecast_v.round(0),
    "forecast_volume_lower95": lower.round(0),
    "forecast_volume_upper95": upper.round(0),
    "forecast_default_rate_pct": forecast_d.round(2),
    "forecast_default_rate_lower95": lower_d.round(2),
    "forecast_default_rate_upper95": upper_d.round(2),
})
forecast_table.to_csv("data/forecast_next_6_months.csv", index=False)
print("\nForecast table:")
print(forecast_table.to_string(index=False))
print("\nBatch 4 (forecasting) complete.")

saved 23_risk_tier_portfolio_composition.png

Forecast table:
forecast_month  forecast_loan_volume  forecast_volume_lower95  forecast_volume_upper95  forecast_default_rate_pct  forecast_default_rate_lower95  forecast_default_rate_upper95
    2026-01-01                1103.0                    988.0                   1219.0                      21.97                          19.80                          24.15
    2026-02-01                1153.0                    989.0                   1316.0                      22.04                          18.96                          25.12
    2026-03-01                1202.0                   1002.0                   1402.0                      22.11                          18.34                          25.88
    2026-04-01                1252.0                   1021.0                   1482.0                      22.18                          17.83                          26.53
    2026-05-01                1301.0                   104

**Forecast table (next 6 months):**

In [71]:
forecast_table = pd.read_csv("data/forecast_next_6_months.csv")
forecast_table

,forecast_month,forecast_loan_volume,forecast_volume_lower95,forecast_volume_upper95,forecast_default_rate_pct,forecast_default_rate_lower95,forecast_default_rate_upper95
0,2026-01-01,1103.0,988.0,1219.0,21.97,19.80,24.15
1,2026-02-01,1153.0,989.0,1316.0,22.04,18.96,25.12
2,2026-03-01,1202.0,1002.0,1402.0,22.11,18.34,25.88
3,2026-04-01,1252.0,1021.0,1482.0,22.18,17.83,26.53
4,2026-05-01,1301.0,1043.0,1559.0,22.25,17.38,27.11
5,2026-06-01,1351.0,1068.0,1633.0,22.32,16.98,27.65


## 5. SQL Analytics Layer

A normalized schema (`sql/01_schema.sql`, PostgreSQL syntax with constraints
and indexes) and 10 dashboard-ready views (`sql/02_dashboard_views.sql`) power
the analytics layer. Below, the same logic is validated end-to-end by loading
the scored dataset into **SQLite** and running SQLite-compatible versions of
every view — proving the SQL actually executes and produces correct output,
not just untested query text.

In [72]:
DATA_PATH = "data/credit_risk_full.csv"
DB_PATH = "sql/credit_risk.db"
OUT_DIR = "sql/query_results"
os.makedirs(OUT_DIR, exist_ok=True)

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

df = pd.read_csv(DATA_PATH)
conn = sqlite3.connect(DB_PATH)
df.to_sql("loans", conn, index=False, if_exists="replace")
print(f"Loaded {len(df):,} rows into SQLite table 'loans'\n")

queries = {
"portfolio_kpis": """
    SELECT
        COUNT(*) AS total_loans,
        ROUND(AVG(loan_amnt),2) AS avg_loan_amount,
        ROUND(SUM(loan_amnt),2) AS total_loan_value,
        ROUND(100.0*SUM(loan_status)/COUNT(*),2) AS default_rate_pct,
        ROUND(AVG(loan_int_rate),2) AS avg_interest_rate,
        ROUND(AVG(person_income),2) AS avg_borrower_income,
        ROUND(AVG(loan_percent_income)*100,2) AS avg_dti_pct,
        ROUND(AVG(predicted_default_probability)*100,2) AS avg_predicted_risk_pct
    FROM loans;
""",
"default_by_grade": """
    SELECT loan_grade, COUNT(*) AS loan_count,
        ROUND(100.0*SUM(loan_status)/COUNT(*),2) AS default_rate_pct,
        ROUND(AVG(loan_amnt),2) AS avg_loan_amount,
        ROUND(AVG(loan_int_rate),2) AS avg_interest_rate,
        ROUND(SUM(loan_amnt),2) AS total_exposure
    FROM loans GROUP BY loan_grade ORDER BY loan_grade;
""",
"default_by_intent": """
    SELECT loan_intent, COUNT(*) AS loan_count,
        ROUND(100.0*SUM(loan_status)/COUNT(*),2) AS default_rate_pct,
        ROUND(AVG(loan_amnt),2) AS avg_loan_amount
    FROM loans GROUP BY loan_intent ORDER BY default_rate_pct DESC;
""",
"risk_matrix_grade_home": """
    SELECT loan_grade, person_home_ownership, COUNT(*) AS loan_count,
        ROUND(100.0*SUM(loan_status)/COUNT(*),2) AS default_rate_pct
    FROM loans GROUP BY loan_grade, person_home_ownership
    ORDER BY loan_grade, person_home_ownership;
""",
"dti_band_analysis": """
    SELECT dti_risk_band, COUNT(*) AS loan_count,
        ROUND(100.0*SUM(loan_status)/COUNT(*),2) AS default_rate_pct,
        ROUND(AVG(loan_int_rate),2) AS avg_interest_rate
    FROM loans GROUP BY dti_risk_band;
""",
"prior_default_impact": """
    SELECT cb_person_default_on_file AS has_prior_default_flag, COUNT(*) AS loan_count,
        ROUND(100.0*SUM(loan_status)/COUNT(*),2) AS current_default_rate_pct
    FROM loans GROUP BY cb_person_default_on_file;
""",
"monthly_trend": """
    SELECT substr(sim_application_date,1,7) AS trend_month,
        COUNT(*) AS loan_volume,
        ROUND(100.0*SUM(loan_status)/COUNT(*),2) AS default_rate_pct,
        ROUND(AVG(loan_amnt),2) AS avg_loan_amount,
        ROUND(SUM(loan_amnt),2) AS total_originated
    FROM loans GROUP BY trend_month ORDER BY trend_month;
""",
"risk_tier_distribution": """
    SELECT predicted_risk_tier, COUNT(*) AS loan_count,
        ROUND(100.0*COUNT(*)/(SELECT COUNT(*) FROM loans),2) AS pct_of_portfolio,
        ROUND(AVG(predicted_default_probability)*100,2) AS avg_predicted_prob_pct,
        ROUND(100.0*SUM(loan_status)/COUNT(*),2) AS actual_default_rate_pct
    FROM loans GROUP BY predicted_risk_tier;
""",
"top_risk_segments": """
    SELECT loan_grade, loan_intent, COUNT(*) AS loan_count,
        ROUND(100.0*SUM(loan_status)/COUNT(*),2) AS default_rate_pct,
        ROUND(SUM(loan_amnt),2) AS total_exposure
    FROM loans GROUP BY loan_grade, loan_intent
    HAVING COUNT(*) >= 30
    ORDER BY default_rate_pct DESC LIMIT 10;
""",
}

for name, q in queries.items():
    result = pd.read_sql_query(q, conn)
    result.to_csv(f"{OUT_DIR}/{name}.csv", index=False)
    print(f"=== {name} ===")
    print(result.to_string(index=False))
    print()

conn.close()
print("All SQL dashboard queries executed successfully and saved to sql/query_results/")


Loaded 32,399 rows into SQLite table 'loans'

=== portfolio_kpis ===
 total_loans  avg_loan_amount  total_loan_value  default_rate_pct  avg_interest_rate  avg_borrower_income  avg_dti_pct  avg_predicted_risk_pct
       32399          9592.87       310799425.0             21.88              11.02             65536.75        17.03                   32.99

=== default_by_grade ===
loan_grade  loan_count  default_rate_pct  avg_loan_amount  avg_interest_rate  total_exposure
         A       10695              9.97          8544.94               7.34      91388150.0
         B       10384             16.32          9990.65              11.00     103742875.0
         C        6433             20.77          9220.62              13.47      59316275.0
         D        3619             59.05         10849.09              15.36      39262850.0
         E         963             64.49         12919.91              16.99      12441875.0
         F         241             70.54         14717.32    

## 6. Executive Dashboard Summary

A single-image KPI mosaic combining headline portfolio numbers and the top
charts, styled like a BI dashboard tile layout.

In [73]:
DATA_PATH = "data/credit_risk_full.csv"
VIZ_DIR = "visuals"

NAVY = "#1B2A4A"
ACCENT = "#E8743B"
GOOD = "#3AA655"
BAD = "#C94C4C"
BLUE = "#2E5E8C"
plt.rcParams["savefig.dpi"] = 160

df = pd.read_csv(DATA_PATH)
order_grade = sorted(df.loan_grade.unique())

total_loans = len(df)
default_rate = df.loan_status.mean()*100
total_exposure = df.loan_amnt.sum()
avg_rate = df.loan_int_rate.mean()
avg_income = df.person_income.mean()

fig = plt.figure(figsize=(20,11))
fig.patch.set_facecolor("white")
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.55, wspace=0.35, height_ratios=[0.6,1,1])

fig.suptitle("CREDIT RISK PORTFOLIO — EXECUTIVE DASHBOARD", fontsize=22, fontweight="bold",
             color=NAVY, x=0.01, ha="left", y=0.99)
fig.text(0.01, 0.955, "Loan-level portfolio analysis  |  32,399 loans  |  Data as of latest snapshot",
          fontsize=11, color="gray", ha="left")

# ---- KPI cards (row 0) ----
kpis = [
    ("TOTAL LOANS", f"{total_loans:,}", BLUE),
    ("DEFAULT RATE", f"{default_rate:.1f}%", BAD),
    ("TOTAL EXPOSURE", f"${total_exposure/1e6:.1f}M", ACCENT),
    ("AVG INTEREST RATE", f"{avg_rate:.1f}%", GOOD),
]
for i, (label, value, color) in enumerate(kpis):
    ax = fig.add_subplot(gs[0, i])
    ax.axis("off")
    ax.add_patch(plt.Rectangle((0,0), 1, 1, transform=ax.transAxes, facecolor="#F4F6F9",
                                 edgecolor=color, linewidth=2.5))
    ax.text(0.5, 0.62, value, ha="center", va="center", fontsize=28, fontweight="bold", color=color, transform=ax.transAxes)
    ax.text(0.5, 0.22, label, ha="center", va="center", fontsize=11.5, color=NAVY, fontweight="bold", transform=ax.transAxes)

# ---- Row 1: default by grade, portfolio mix, risk tier ----
ax1 = fig.add_subplot(gs[1, 0:2])
grade_default = df.groupby("loan_grade")["loan_status"].mean().reindex(order_grade)*100
bars = ax1.bar(grade_default.index, grade_default.values, color=ACCENT)
for b,v in zip(bars, grade_default.values):
    ax1.text(b.get_x()+b.get_width()/2, v+1, f"{v:.0f}%", ha="center", fontsize=9, fontweight="bold")
ax1.set_title("Default Rate by Loan Grade", fontweight="bold", color=NAVY, loc="left")
ax1.set_ylabel("Default Rate (%)")

ax2 = fig.add_subplot(gs[1, 2])
tier_order = ["Very Low","Low","Moderate","High","Very High"]
tier_counts = df["predicted_risk_tier"].value_counts().reindex(tier_order)
colors_tier = [GOOD, "#79A968", "#D4AC0D", ACCENT, BAD]
ax2.pie(tier_counts.values, colors=colors_tier, autopct="%1.0f%%", pctdistance=0.78,
        wedgeprops=dict(width=0.42), startangle=90, textprops={"fontsize":9})
ax2.set_title("Portfolio Risk Tier Mix", fontweight="bold", color=NAVY, loc="left")

ax3 = fig.add_subplot(gs[1, 3])
intent_default = df.groupby("loan_intent")["loan_status"].mean().sort_values(ascending=True)*100
ax3.barh(intent_default.index, intent_default.values, color=BLUE)
ax3.set_title("Default % by Purpose", fontweight="bold", color=NAVY, loc="left", fontsize=11)
ax3.tick_params(labelsize=8)

# ---- Row 2: monthly trend, DTI band, home ownership ----
monthly = df.copy()
monthly["month"] = pd.to_datetime(monthly.sim_application_date).dt.to_period("M").dt.to_timestamp()
trend = monthly.groupby("month").agg(vol=("loan_status","size"), dr=("loan_status","mean")).reset_index()

ax4 = fig.add_subplot(gs[2, 0:2])
ax4b = ax4.twinx()
ax4.bar(trend.month, trend.vol, width=20, color="#C9D6E3", label="Volume")
ax4b.plot(trend.month, trend.dr*100, color=BAD, linewidth=2.2, marker="o", markersize=3, label="Default Rate")
ax4.set_title("Monthly Volume & Default Rate Trend (simulated cohort)", fontweight="bold", color=NAVY, loc="left", fontsize=11)
ax4.set_ylabel("Loan Volume", fontsize=9)
ax4b.set_ylabel("Default Rate (%)", fontsize=9, color=BAD)
ax4.tick_params(axis='x', labelsize=8, rotation=30)

ax5 = fig.add_subplot(gs[2, 2])
dti_order = ["Very Low","Low","Moderate","High","Very High"]
dti_default = df.groupby("dti_risk_band", observed=True)["loan_status"].mean().reindex(dti_order)*100
ax5.bar(dti_default.index, dti_default.values, color=GOOD)
ax5.set_title("Default % by DTI Band", fontweight="bold", color=NAVY, loc="left", fontsize=11)
ax5.tick_params(axis='x', labelsize=7, rotation=30)

ax6 = fig.add_subplot(gs[2, 3])
home_default = df.groupby("person_home_ownership")["loan_status"].mean().sort_values(ascending=False)*100
ax6.bar(home_default.index, home_default.values, color="#8E6FBE")
ax6.set_title("Default % by Home Status", fontweight="bold", color=NAVY, loc="left", fontsize=11)
ax6.tick_params(axis='x', labelsize=8)

fig.savefig(f"{VIZ_DIR}/00_executive_dashboard.png", bbox_inches="tight", facecolor="white")
plt.close(fig)
print("saved 00_executive_dashboard.png")


saved 00_executive_dashboard.png
